In [2]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [3]:
crime_path = Path("../data/raw/nypd_crime.csv")

crime_df = pd.read_csv(crime_path)

print("Rows:", len(crime_df))
print("Columns:", crime_df.columns.tolist())

Rows: 500000
Columns: ['cmplnt_num', 'cmplnt_fr_dt', 'cmplnt_fr_tm', 'cmplnt_to_dt', 'cmplnt_to_tm', 'addr_pct_cd', 'rpt_dt', 'ky_cd', 'ofns_desc', 'pd_cd', 'pd_desc', 'crm_atpt_cptd_cd', 'law_cat_cd', 'boro_nm', 'loc_of_occur_desc', 'prem_typ_desc', 'juris_desc', 'jurisdiction_code', 'parks_nm', 'hadevelopt', 'housing_psa', 'x_coord_cd', 'y_coord_cd', 'susp_age_group', 'susp_race', 'susp_sex', 'transit_district', 'latitude', 'longitude', 'lat_lon', 'patrol_boro', 'station_name', 'vic_age_group', 'vic_race', 'vic_sex']


In [4]:
crime_df[
    [
        "cmplnt_fr_dt",
        "cmplnt_fr_tm",
        "ofns_desc",
        "latitude",
        "longitude"
    ]
].head()

,cmplnt_fr_dt,cmplnt_fr_tm,ofns_desc,latitude,longitude
0,2025-12-31T00:00:00.000,00:23:00,CANNABIS RELATED OFFENSES,40.665249,-73.788948
1,2025-12-31T00:00:00.000,07:10:00,FRAUDS,40.665249,-73.788948
2,2025-09-29T00:00:00.000,11:15:00,RAPE,40.766996,-73.963922
3,2025-12-31T00:00:00.000,01:00:00,BURGLARY,40.648106,-73.945397
4,2025-12-31T00:00:00.000,17:00:00,HARRASSMENT 2,40.686770,-73.822818


In [5]:
required_columns = [
    "cmplnt_fr_dt",
    "cmplnt_fr_tm",
    "latitude",
    "longitude"
]

crime_df = crime_df.dropna(subset=required_columns)

print("Rows after removing missing values:", len(crime_df))

Rows after removing missing values: 500000


In [6]:
crime_df["crime_datetime"] = pd.to_datetime(
    crime_df["cmplnt_fr_dt"].astype(str) + " " + crime_df["cmplnt_fr_tm"].astype(str),
    errors="coerce"
)

print(crime_df["crime_datetime"].head())

C:\Users\SIC\AppData\Local\Temp\ipykernel_23344\2854398484.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_df["crime_datetime"] = pd.to_datetime(


0   2025-12-31 00:23:00
1   2025-12-31 07:10:00
2   2025-09-29 11:15:00
3   2025-12-31 01:00:00
4   2025-12-31 17:00:00
Name: crime_datetime, dtype: datetime64[us]


In [7]:
invalid_datetime = crime_df["crime_datetime"].isna().sum()

print("Invalid crime_datetime values:", invalid_datetime)

Invalid crime_datetime values: 0


In [8]:
crime_df[
    [
        "cmplnt_fr_dt",
        "cmplnt_fr_tm",
        "crime_datetime"
    ]
].head(10)

,cmplnt_fr_dt,cmplnt_fr_tm,crime_datetime
0,2025-12-31T00:00:00.000,00:23:00,2025-12-31 00:23:00
1,2025-12-31T00:00:00.000,07:10:00,2025-12-31 07:10:00
2,2025-09-29T00:00:00.000,11:15:00,2025-09-29 11:15:00
3,2025-12-31T00:00:00.000,01:00:00,2025-12-31 01:00:00
4,2025-12-31T00:00:00.000,17:00:00,2025-12-31 17:00:00
5,2025-12-22T00:00:00.000,21:50:00,2025-12-22 21:50:00
6,2025-12-31T00:00:00.000,10:00:00,2025-12-31 10:00:00
7,2025-12-31T00:00:00.000,10:15:00,2025-12-31 10:15:00
8,2025-12-30T00:00:00.000,08:00:00,2025-12-30 08:00:00
9,2025-12-26T00:00:00.000,09:30:00,2025-12-26 09:30:00


In [9]:
allowed_offenses = [
    "ROBBERY",
    "BURGLARY",
    "GRAND LARCENY",
    "ASSAULT",
    "PETIT LARCENY",
    "MISCHIEF"
]

print(allowed_offenses)

['ROBBERY', 'BURGLARY', 'GRAND LARCENY', 'ASSAULT', 'PETIT LARCENY', 'MISCHIEF']


In [10]:
crime_df = crime_df[
    crime_df["ofns_desc"].isin(allowed_offenses)
].copy()

print("Rows after offense filtering:", len(crime_df))

Rows after offense filtering: 156898


In [11]:
crime_df["ofns_desc"].value_counts()

ofns_desc
PETIT LARCENY    90614
GRAND LARCENY    42165
ROBBERY          13223
BURGLARY         10896
Name: count, dtype: int64

In [12]:
print("Unique offense categories:", crime_df["ofns_desc"].unique())

Unique offense categories: <ArrowStringArray>
['BURGLARY', 'PETIT LARCENY', 'GRAND LARCENY', 'ROBBERY']
Length: 4, dtype: str


In [13]:
crime_df["ofns_desc"].value_counts().head(30)

ofns_desc
PETIT LARCENY    90614
GRAND LARCENY    42165
ROBBERY          13223
BURGLARY         10896
Name: count, dtype: int64

In [14]:
print(
    crime_df[crime_df["ofns_desc"].isin(["ASSAULT", "MISCHIEF"])]["ofns_desc"]
    .value_counts()
)

Series([], Name: count, dtype: int64)


In [15]:
print("ALL RAW OFFENSE CATEGORIES:")
print(crime_df["ofns_desc"].value_counts())

ALL RAW OFFENSE CATEGORIES:
ofns_desc
PETIT LARCENY    90614
GRAND LARCENY    42165
ROBBERY          13223
BURGLARY         10896
Name: count, dtype: int64


In [16]:
crime_df[["cmplnt_fr_dt", "cmplnt_fr_tm", "crime_datetime"]].head()

,cmplnt_fr_dt,cmplnt_fr_tm,crime_datetime
3,2025-12-31T00:00:00.000,01:00:00,2025-12-31 01:00:00
7,2025-12-31T00:00:00.000,10:15:00,2025-12-31 10:15:00
14,2025-12-31T00:00:00.000,01:55:00,2025-12-31 01:55:00
15,2025-12-30T00:00:00.000,21:00:00,2025-12-30 21:00:00
24,2025-12-31T00:00:00.000,13:12:00,2025-12-31 13:12:00


In [17]:
crime_df["crime_time"] = crime_df["crime_datetime"].dt.time

crime_df["is_nighttime"] = (
    (crime_df["crime_datetime"].dt.hour >= 18) |
    (crime_df["crime_datetime"].dt.hour <= 6)
)

crime_df["is_nighttime"].value_counts()

is_nighttime
False    93704
True     63194
Name: count, dtype: int64

In [18]:
crime_df = crime_df[
    crime_df["is_nighttime"] == True
].copy()

print("Rows after nighttime filtering:", len(crime_df))

Rows after nighttime filtering: 63194


In [19]:
crime_df["is_nighttime"].value_counts()

is_nighttime
True    63194
Name: count, dtype: int64

In [20]:
crime_df = crime_df[
    crime_df["latitude"].between(40.49, 40.92)
    &
    crime_df["longitude"].between(-74.26, -73.69)
].copy()

print("Rows after NYC coordinate filtering:", len(crime_df))

Rows after NYC coordinate filtering: 63194


In [21]:
print("Latitude range:")
print(crime_df["latitude"].min(), crime_df["latitude"].max())

print("\nLongitude range:")
print(crime_df["longitude"].min(), crime_df["longitude"].max())

Latitude range:
40.499625 40.912722

Longitude range:
-74.254439 -73.700719


In [22]:
crime_gdf = gpd.GeoDataFrame(
    crime_df,
    geometry=gpd.points_from_xy(
        crime_df["longitude"],
        crime_df["latitude"]
    ),
    crs="EPSG:4326"
)

In [23]:
print(crime_gdf.head())

   cmplnt_num             cmplnt_fr_dt cmplnt_fr_tm             cmplnt_to_dt  \
3   318184162  2025-12-31T00:00:00.000     01:00:00  2025-12-31T00:00:00.000   
14  318190126  2025-12-31T00:00:00.000     01:55:00  2025-12-31T00:00:00.000   
15  318202723  2025-12-30T00:00:00.000     21:00:00  2025-12-31T00:00:00.000   
25  318184194  2025-12-30T00:00:00.000     19:00:00  2025-12-31T00:00:00.000   
26  318198652  2025-12-29T00:00:00.000     20:00:00  2025-12-31T00:00:00.000   

   cmplnt_to_tm  addr_pct_cd                   rpt_dt  ky_cd      ofns_desc  \
3      06:00:00           67  2025-12-31T00:00:00.000    107       BURGLARY   
14     01:57:00           72  2025-12-31T00:00:00.000    341  PETIT LARCENY   
15     13:00:00           81  2025-12-31T00:00:00.000    109  GRAND LARCENY   
25     07:20:00          101  2025-12-31T00:00:00.000    107       BURGLARY   
26     12:30:00           60  2025-12-31T00:00:00.000    341  PETIT LARCENY   

    pd_cd  ...                      lat_lon 

In [24]:
print(crime_gdf.crs)

EPSG:4326


In [25]:
print(crime_gdf.geometry.head())

3      POINT (-73.9454 40.64811)
14    POINT (-74.01203 40.64524)
15    POINT (-73.93574 40.68643)
25    POINT (-73.75115 40.60228)
26    POINT (-73.97644 40.57727)
Name: geometry, dtype: geometry


In [26]:
print("Final rows:", len(crime_gdf))

print("\nOffense categories:")
print(crime_gdf["ofns_desc"].value_counts())

print("\nNighttime values:")
print(crime_gdf["is_nighttime"].value_counts())

print("\nMissing values:")
print(
    crime_gdf[
        ["crime_datetime", "latitude", "longitude", "ofns_desc"]
    ].isna().sum()
)

print("\nCRS:")
print(crime_gdf.crs)

Final rows: 63194

Offense categories:
ofns_desc
PETIT LARCENY    31472
GRAND LARCENY    18257
ROBBERY           7409
BURGLARY          6056
Name: count, dtype: int64

Nighttime values:
is_nighttime
True    63194
Name: count, dtype: int64

Missing values:
crime_datetime    0
latitude          0
longitude         0
ofns_desc         0
dtype: int64

CRS:
EPSG:4326


In [27]:
crime_gdf.to_parquet(
    "../data/processed/clean_crime.parquet",
    index=False
)

In [28]:
# Verification

test_crime = gpd.read_parquet(
    "../data/processed/clean_crime.parquet"
)

print(test_crime.shape)
print(test_crime.head())
print(test_crime.crs)


(63194, 39)
  cmplnt_num             cmplnt_fr_dt cmplnt_fr_tm             cmplnt_to_dt  \
0  318184162  2025-12-31T00:00:00.000     01:00:00  2025-12-31T00:00:00.000   
1  318190126  2025-12-31T00:00:00.000     01:55:00  2025-12-31T00:00:00.000   
2  318202723  2025-12-30T00:00:00.000     21:00:00  2025-12-31T00:00:00.000   
3  318184194  2025-12-30T00:00:00.000     19:00:00  2025-12-31T00:00:00.000   
4  318198652  2025-12-29T00:00:00.000     20:00:00  2025-12-31T00:00:00.000   

  cmplnt_to_tm  addr_pct_cd                   rpt_dt  ky_cd      ofns_desc  \
0     06:00:00           67  2025-12-31T00:00:00.000    107       BURGLARY   
1     01:57:00           72  2025-12-31T00:00:00.000    341  PETIT LARCENY   
2     13:00:00           81  2025-12-31T00:00:00.000    109  GRAND LARCENY   
3     07:20:00          101  2025-12-31T00:00:00.000    107       BURGLARY   
4     12:30:00           60  2025-12-31T00:00:00.000    341  PETIT LARCENY   

   pd_cd  ...                      lat_lon  

In [29]:
print(
    crime_df[
        ["cmplnt_fr_dt", "cmplnt_fr_tm"]
    ].head(20).to_string(index=False)
)

           cmplnt_fr_dt cmplnt_fr_tm
2025-12-31T00:00:00.000     01:00:00
2025-12-31T00:00:00.000     01:55:00
2025-12-30T00:00:00.000     21:00:00
2025-12-30T00:00:00.000     19:00:00
2025-12-29T00:00:00.000     20:00:00
2025-12-07T00:00:00.000     05:48:00
2025-12-29T00:00:00.000     20:00:00
2025-12-31T00:00:00.000     19:00:00
2025-12-31T00:00:00.000     04:42:00
2025-12-29T00:00:00.000     05:49:00
2025-12-31T00:00:00.000     18:17:00
2025-12-31T00:00:00.000     06:24:00
2025-12-28T00:00:00.000     05:00:00
2025-12-29T00:00:00.000     18:18:00
2025-07-19T00:00:00.000     00:01:00
2025-12-30T00:00:00.000     19:30:00
2025-12-29T00:00:00.000     20:45:00
2025-12-30T00:00:00.000     22:15:00
2025-12-31T00:00:00.000     06:35:00
2025-12-28T00:00:00.000     21:00:00


In [30]:
print(crime_df["cmplnt_fr_dt"].head(20).tolist())

['2025-12-31T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-30T00:00:00.000', '2025-12-30T00:00:00.000', '2025-12-29T00:00:00.000', '2025-12-07T00:00:00.000', '2025-12-29T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-29T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-28T00:00:00.000', '2025-12-29T00:00:00.000', '2025-07-19T00:00:00.000', '2025-12-30T00:00:00.000', '2025-12-29T00:00:00.000', '2025-12-30T00:00:00.000', '2025-12-31T00:00:00.000', '2025-12-28T00:00:00.000']


In [31]:
print(crime_df["cmplnt_fr_tm"].head(20).tolist())

['01:00:00', '01:55:00', '21:00:00', '19:00:00', '20:00:00', '05:48:00', '20:00:00', '19:00:00', '04:42:00', '05:49:00', '18:17:00', '06:24:00', '05:00:00', '18:18:00', '00:01:00', '19:30:00', '20:45:00', '22:15:00', '06:35:00', '21:00:00']


In [32]:
print("Minimum date:", crime_df["cmplnt_fr_dt"].min())
print("Maximum date:", crime_df["cmplnt_fr_dt"].max())

Minimum date: 1015-05-01T00:00:00.000
Maximum date: 2025-12-31T00:00:00.000


In [33]:
import os

print(
    "CSV size in MB:",
    round(
        os.path.getsize("../data/raw/nypd_crime.csv") / (1024 * 1024),
        2
    )
)

CSV size in MB: 213.08


In [34]:
print("Rows in crime_df:", len(crime_df))

Rows in crime_df: 63194


In [36]:
raw_check = pd.read_csv(
    "../data/raw/nypd_crime.csv",
    usecols=["cmplnt_fr_dt", "cmplnt_fr_tm"]
)

print(raw_check.dtypes)

print("\nFirst 20 rows:")
print(raw_check.head(20).to_string(index=False))

cmplnt_fr_dt    str
cmplnt_fr_tm    str
dtype: object

First 20 rows:
           cmplnt_fr_dt cmplnt_fr_tm
2025-12-31T00:00:00.000     00:23:00
2025-12-31T00:00:00.000     07:10:00
2025-09-29T00:00:00.000     11:15:00
2025-12-31T00:00:00.000     01:00:00
2025-12-31T00:00:00.000     17:00:00
2025-12-22T00:00:00.000     21:50:00
2025-12-31T00:00:00.000     10:00:00
2025-12-31T00:00:00.000     10:15:00
2025-12-30T00:00:00.000     08:00:00
2025-12-26T00:00:00.000     09:30:00
2025-12-31T00:00:00.000     01:53:00
2025-12-30T00:00:00.000     03:00:00
2025-12-31T00:00:00.000     03:10:00
2025-12-30T00:00:00.000     19:00:00
2025-12-31T00:00:00.000     01:55:00
2025-12-30T00:00:00.000     21:00:00
2025-12-31T00:00:00.000     17:20:00
2025-12-31T00:00:00.000     05:45:00
2025-12-31T00:00:00.000     07:30:00
2025-12-30T00:00:00.000     18:41:00


In [37]:
raw_check["cmplnt_fr_dt_parsed"] = pd.to_datetime(
    raw_check["cmplnt_fr_dt"],
    errors="coerce",
    format="mixed"
)

print(
    raw_check["cmplnt_fr_dt_parsed"]
    .sort_values()
    .head(20)
)

391971   1014-04-20
390311   1015-05-01
108303   1015-09-02
75678    1015-11-09
68432    1015-11-14
34391    1015-12-06
384339   1025-05-05
70748    1025-11-13
23192    1025-12-06
25985    1025-12-12
16248    1025-12-19
329725   1963-05-01
467594   1970-01-01
465563   1975-03-16
385156   1975-04-18
351571   1975-05-24
334338   1975-06-04
204648   1975-06-20
203961   1975-08-23
154929   1975-08-30
Name: cmplnt_fr_dt_parsed, dtype: datetime64[us]


In [38]:
raw_check[
    raw_check["cmplnt_fr_dt_parsed"].dt.year < 1900
].head(20)

,cmplnt_fr_dt,cmplnt_fr_tm,cmplnt_fr_dt_parsed
16248,1025-12-19T00:00:00.000,05:35:00,1025-12-19
23192,1025-12-06T00:00:00.000,20:15:00,1025-12-06
25985,1025-12-12T00:00:00.000,15:38:00,1025-12-12
34391,1015-12-06T00:00:00.000,18:00:00,1015-12-06
68432,1015-11-14T00:00:00.000,13:48:00,1015-11-14
70748,1025-11-13T00:00:00.000,08:30:00,1025-11-13
75678,1015-11-09T00:00:00.000,20:00:00,1015-11-09
108303,1015-09-02T00:00:00.000,08:00:00,1015-09-02
384339,1025-05-05T00:00:00.000,15:02:00,1025-05-05
390311,1015-05-01T00:00:00.000,20:50:00,1015-05-01


In [39]:
print(
    "Number of dates before 1900:",
    (
        raw_check["cmplnt_fr_dt_parsed"].dt.year < 1900
    ).sum()
)

Number of dates before 1900: 11


In [40]:
print(
    "Raw minimum:",
    raw_check["cmplnt_fr_dt_parsed"].min()
)

print(
    "Raw maximum:",
    raw_check["cmplnt_fr_dt_parsed"].max()
)

Raw minimum: 1014-04-20 00:00:00
Raw maximum: 2025-12-31 00:00:00


In [41]:
print("Raw CSV rows:", len(raw_check))

Raw CSV rows: 500000


In [42]:
print(raw_check.dtypes)

cmplnt_fr_dt                      str
cmplnt_fr_tm                      str
cmplnt_fr_dt_parsed    datetime64[us]
dtype: object


In [43]:
print(
    raw_check[
        raw_check["cmplnt_fr_dt_parsed"].dt.year < 1900
    ].head(20)
)

                   cmplnt_fr_dt cmplnt_fr_tm cmplnt_fr_dt_parsed
16248   1025-12-19T00:00:00.000     05:35:00          1025-12-19
23192   1025-12-06T00:00:00.000     20:15:00          1025-12-06
25985   1025-12-12T00:00:00.000     15:38:00          1025-12-12
34391   1015-12-06T00:00:00.000     18:00:00          1015-12-06
68432   1015-11-14T00:00:00.000     13:48:00          1015-11-14
70748   1025-11-13T00:00:00.000     08:30:00          1025-11-13
75678   1015-11-09T00:00:00.000     20:00:00          1015-11-09
108303  1015-09-02T00:00:00.000     08:00:00          1015-09-02
384339  1025-05-05T00:00:00.000     15:02:00          1025-05-05
390311  1015-05-01T00:00:00.000     20:50:00          1015-05-01
391971  1014-04-20T00:00:00.000     19:49:00          1014-04-20


In [44]:
print("Raw CSV rows:", len(raw_check))
print("Raw minimum:", raw_check["cmplnt_fr_dt_parsed"].min())
print("Raw maximum:", raw_check["cmplnt_fr_dt_parsed"].max())

Raw CSV rows: 500000
Raw minimum: 1014-04-20 00:00:00
Raw maximum: 2025-12-31 00:00:00
